In [ ]:

import sys; sys.path.insert(0, '../src')
import numpy as np
from models.graphene import SingleLayerGrapheneTB, SingleLayerGrapheneKP
from response.dos import (compute_jdos, compute_optical_jdos,
                           compute_eigenvalues, generate_dirac_mesh, dirac_jdos_analytical,
                           dirac_optical_jdos_analytical)
from response.polarization import generate_k_mesh

vF = np.sqrt(3)/2 * 2.78; t = 2.78; tb = SingleLayerGrapheneTB(t=t)


In [ ]:
# ── 2. KP Dirac JDOS ──────────────────────────────
print()
print('=== 2. KP Dirac-mesh JDOS ===')
kp = SingleLayerGrapheneKP(t=t, valley=+1)
k_mesh = generate_dirac_mesh(kp, nk=120, k_cut=0.5)
area = (2*0.5)**2
kp.valley_degeneracy = 2
w_kp, jd_kp = compute_jdos(kp, nw=300, sigma=0.015,
                            w_range=(0, 2.0), k_cart=k_mesh, k_area=area)
kp.valley_degeneracy = 1
mask_k = (w_kp > 0.05) & (w_kp < 1.5)
sl_kp = np.polyfit(w_kp[mask_k], jd_kp[mask_k], 1)[0]

sl_an = dirac_jdos_analytical(1.0, vF)

print(f'  Dirac slope: {sl_kp:.4f} (analytical: {sl_an:.4f}), ratio={sl_kp/sl_an:.3f}')
print(f'  JDOS(0)={jd_kp[0]:.6f}')

In [ ]:

import matplotlib.pyplot as plt

plt.style.use('bmh')

%matplotlib inline
%config InlineBackend.figure_format='retina'
import matplotlib as mpl
mpl.rcParams.update({
    'font.size': 10.5,
    # 'font.family': ['Times New Roman','SimSun'],# 'HP Simplified Hans', # Tahoma
    'axes.unicode_minus': False,
    'axes.facecolor': 'white',
    'figure.facecolor': 'white',
    'savefig.facecolor': 'white', # optional: saved figures
    # 'grid.color': '#e0e0e0', # optional: lighter grid
    'axes.edgecolor': 'black', # optional: clearer axes
})

fig, ax = plt.subplots(figsize=(4,3))
ax.plot(w_kp, jd_kp) # 3.3e7 for i=31; 8e6 for i=15
ax.set_xbound(w_kp[0], w_kp[-1])
ax.set_xlabel(r"Energy (eV)") # $\gamma_0$
ax.set_ylabel("JDOS") #  (states/eV·unit cell)
ax.set_title(''.join([r'$\eta$=', f'{0.015:.2g} ',r'eV; $\Delta E$=',f'{2.0/300:.3g} ','eV']), loc='right') #$\gamma_0$
fig.tight_layout()
plt.show()

In [ ]:
# 统一网格
_, kc = generate_k_mesh(120, tb.reciprocal_vectors)
Ek, Vk = compute_eigenvalues(tb, kc)
# JDOS
w_j, jd = compute_jdos(tb, sigma=0.12, w_range=(0,8), nw=400, k_cart=kc, E_k=Ek)
idx_j = np.argmax(jd)
print(f'JDOS  VHS at w={w_j[idx_j]:.2f} eV (expect ~{2*t})')
m_j = (w_j>0.05)&(w_j<1.5)
s_j = np.polyfit(w_j[m_j], jd[m_j], 1)[0]
sa_j = dirac_jdos_analytical(1.0, vF)
print(f'JDOS  Dirac slope: {s_j:.4f} (analyt: {sa_j:.4f}), ratio={s_j/sa_j:.3f}')
# O-JDOS
w_o, oj = compute_optical_jdos(tb, sigma=0.12, w_range=(0,8), nw=400, k_cart=kc, E_k=Ek, V_k=Vk)
idx_o = np.argmax(oj)
print(f'O-JDOS VHS at w={w_o[idx_o]:.2f} eV')
m_o = (w_o>0.05)&(w_o<1.5)
s_o = np.polyfit(w_o[m_o], oj[m_o], 1)[0]
sa_o = dirac_optical_jdos_analytical(1.0, vF)
print(f'O-JDOS Dirac slope: {s_o:.4f} (analyt: {sa_o:.4f}), ratio={s_o/sa_o:.3f}')
# O-JDOS/JDOS
r = oj[m_o] / (jd[m_j]+1e-12)
print(f'<O-JDOS/JDOS>: {np.mean(r):.4f} (expect vF^2/2={vF**2/2:.4f})')


In [ ]:

fig, ax = plt.subplots(figsize=(4,3))
ax.plot(w_j, jd, label='JDOS') # 3.3e7 for i=31; 8e6 for i=15
ax.plot(w_o, oj, label='O-JDOS')
ax.set_xbound(w_j[0], w_j[-1])
ax.set_xlabel(r"Energy (eV)") # $\gamma_0$
ax.set_ylabel("JDOS") #  (states/eV·unit cell)
ax.set_title(''.join([r'$\eta$=', f'{0.12:.2g} ',r'eV; $\Delta E$=',f'{8/400:.3g} ','eV']), loc='right') #$\gamma_0$
ax.legend()
fig.tight_layout()
fig.savefig('Graphene-Joint-Density-of-States.png', dpi=200)
plt.show()

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
from models.graphene import SingleLayerGrapheneTB
from response.dos import compute_jdos, compute_optical_jdos, compute_eigenvalues, dirac_jdos_analytical, dirac_optical_jdos_analytical
from response.polarization import generate_k_mesh

vF=np.sqrt(3)/2*2.78; tb=SingleLayerGrapheneTB(t=2.78)
_,kc=generate_k_mesh(200, tb.reciprocal_vectors)
Ek,Vk=compute_eigenvalues(tb,kc)
for sigma in [0.12]:
    w_j,jd=compute_jdos(tb,sigma=sigma,w_range=(0,8),nw=400,k_cart=kc,E_k=Ek)
    w_o,oj=compute_optical_jdos(tb,sigma=sigma,w_range=(0,8),nw=400,k_cart=kc,E_k=Ek,V_k=Vk)
    m=(w_j>0.1)&(w_j<1.5)
    sj=np.polyfit(w_j[m],jd[m],1)[0]; so=np.polyfit(w_o[m],oj[m],1)[0]
    sa_j=dirac_jdos_analytical(1.0,vF); sa_o=dirac_optical_jdos_analytical(1.0,vF)
    r=np.mean(oj[m]/(jd[m]+1e-12))
    print(f'sigma={sigma}: JDOS ratio={sj/sa_j:.3f}, O-JDOS ratio={so/sa_o:.3f}, O/J={r:.4f}')
    fig, ax = plt.subplots(figsize=(4,3))
    ax.plot(w_j, jd) # 3.3e7 for i=31; 8e6 for i=15
    ax.plot(w_o, oj)
    ax.set_xbound(w_j[0], w_j[-1])
    ax.set_xlabel(r"Energy (eV)") # $\gamma_0$
    ax.set_ylabel("JDOS") #  (states/eV·unit cell)
    ax.set_title(''.join([r'$\eta$=', f'{sigma:.2g} ',r'eV; $\Delta E$=',f'{8/400:.3g} ','eV']), loc='right') #$\gamma_0$
    fig.tight_layout()
    plt.show()



## 总结：`src/response/dos.py` 新增的三组函数

```
compute_dos()          —— 态密度 DOS(E)
compute_jdos()         —— 联合态密度 JDOS(ω)
compute_optical_jdos() —— 光学联合态密度 O-JDOS(ω)（速度矩阵元加权）
```

### 调用示例

```python
from response.dos import compute_jdos, compute_optical_jdos, compute_eigenvalues
from response.polarization import generate_k_mesh

# 预对角化（避免 JDOS 和 O-JDOS 各算一遍）
_, kc = generate_k_mesh(200, model.reciprocal_vectors)
Ek, Vk = compute_eigenvalues(model, kc)

# JDOS：纯能带差计数
w, jd = compute_jdos(model, sigma=0.02, k_cart=kc, E_k=Ek)

# O-JDOS：用 |v_{mn}|² 加权 → 接近真实光学响应
w, oj = compute_optical_jdos(model, sigma=0.02, k_cart=kc, E_k=Ek, V_k=Vk)
```

### 验证依据

对于 Dirac 锥，三个量在低能区都有解析预言：

| 量 | 低能行为 | 验证 |
|---|---|---|
| DOS | `g·\|E\|/(2πvF²)` | ratio ~1.01 |
| JDOS | `g·ω/(8πvF²)` | ratio ~1.03 |
| O-JDOS | `g·ω/(16π)` | ratio ~1.03 |
| O-JDOS/JDOS | `vF²/2` | ~2.91 vs 2.90 |